In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import json
import gc
from tqdm.notebook import tqdm

WEIGHTS_PATH = '/kaggle/input/hybrid-model/tensorflow2/default/1/hybrid_model.weights.h5' 

TEST_DATA_DIR = '/kaggle/input/461054610546105/test_landmark_files'
LABELS_PATH = '/kaggle/input/461054610546105/labels.parqet'
MAP_PATH = '/kaggle/input/461054610546105/sign_to_prediction_index_map.json'

# Model Config
INPUT_SIZE = 128
N_COLS = 3       # x, y, z
N_COLS_FINAL = 594 
NUM_CLASSES = 250
ROWS_PER_FRAME = 543

2026-02-08 06:59:06.475882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770533946.650357      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770533946.699236      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770533947.114676      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770533947.114726      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770533947.114729      55 computation_placer.cc:177] computation placer alr

✅ Weights found at /kaggle/input/hybrid-model/tensorflow2/default/1/hybrid_model.weights.h5


In [2]:
def load_relevant_data_subset(pq_path):
    data_columns = ['x', 'y', 'z']
    data = pd.read_parquet(pq_path, columns=data_columns)
    n_frames = int(len(data) / ROWS_PER_FRAME)
    data = data.values.reshape(n_frames, ROWS_PER_FRAME, len(data_columns))
    return data.astype(np.float32)

def get_test_metadata(labels_path, map_path):
    # Load Labels
    if not os.path.exists(labels_path):
        raise FileNotFoundError(f"Labels not found at {labels_path}")
    
    df = pd.read_parquet(labels_path)
    
    # Load Map
    if not os.path.exists(map_path):
        raise FileNotFoundError(f"Map not found at {map_path}")
        
    with open(map_path, 'r') as f:
        sign_map = json.load(f)
        
    # Map signs to integers
    df['sign_ord'] = df['sign'].map(sign_map)
    
    # Filter out any signs that might not be in the map (safety check)
    df = df.dropna(subset=['sign_ord'])
    df['sign_ord'] = df['sign_ord'].astype(int)
    
    return df, sign_map

In [3]:
# LAYERS
class EcaLayer(tf.keras.layers.Layer):
    def __init__(self, kernel_size=5, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = tf.keras.layers.Conv1D(1, kernel_size=kernel_size, padding='same', use_bias=False)

    def call(self, x):
        attn = tf.reduce_mean(x, axis=1, keepdims=True)
        attn = tf.transpose(attn, (0, 2, 1))
        attn = self.conv(attn)
        attn = tf.transpose(attn, (0, 2, 1))
        attn = tf.math.sigmoid(attn)
        return x * attn

class Conv1DBlock(tf.keras.layers.Layer):
    def __init__(self, dim, kernel_size=11, drop_rate=0.2, expand=4):
        super().__init__()
        self.conv = tf.keras.layers.DepthwiseConv1D(kernel_size, padding='same', use_bias=False)
        self.bn = tf.keras.layers.BatchNormalization()
        self.act = tf.keras.layers.Activation('swish')
        self.se = EcaLayer(kernel_size=5)
        self.project = tf.keras.layers.Dense(dim, use_bias=False)
        self.drop = tf.keras.layers.Dropout(drop_rate)
        
    def call(self, x, training=None):
        skip = x
        x = self.conv(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.se(x)
        x = self.project(x)
        if training:
            x = self.drop(x)
        return x + skip

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output) 

class LearnablePositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.pos_embedding = tf.keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        max_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        return x + self.pos_embedding(positions)

class MaskingLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(MaskingLayer, self).__init__(**kwargs)
    
    def call(self, inputs):
        frames, non_empty_frame_idxs = inputs
        mask = tf.math.not_equal(non_empty_frame_idxs, -1)
        mask = tf.cast(mask, dtype=frames.dtype)
        mask_expanded = tf.expand_dims(mask, -1)
        return frames * mask_expanded

# --- PREPROCESSING (Must match Training exactly) ---
# Landmark indices
LIPS_IDXS0 = np.array([
        61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
        291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
        95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    ])
LEFT_HAND_IDXS0 = np.arange(468,489)
RIGHT_HAND_IDXS0 = np.arange(522,543)
LEFT_POSE_IDXS0 = np.array([502, 504, 506, 508, 510])
RIGHT_POSE_IDXS0 = np.array([503, 505, 507, 509, 511])
LANDMARK_IDXS_LEFT_DOMINANT0 = np.concatenate((LIPS_IDXS0, LEFT_HAND_IDXS0, LEFT_POSE_IDXS0))
LANDMARK_IDXS_RIGHT_DOMINANT0 = np.concatenate((LIPS_IDXS0, RIGHT_HAND_IDXS0, RIGHT_POSE_IDXS0))
HAND_IDXS0 = np.concatenate((LEFT_HAND_IDXS0, RIGHT_HAND_IDXS0), axis=0)

# Processed Indices
LIPS_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LIPS_IDXS0)).squeeze()
LEFT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_HAND_IDXS0)).squeeze()
RIGHT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, RIGHT_HAND_IDXS0)).squeeze()
POSE_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_POSE_IDXS0)).squeeze()

class PreprocessLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(PreprocessLayer, self).__init__() 
        self.lips_idxs = LIPS_IDXS
        self.left_hand_idxs = LEFT_HAND_IDXS
        self.pose_idxs = POSE_IDXS
        self.landmark_idxs_left = LANDMARK_IDXS_LEFT_DOMINANT0
        self.landmark_idxs_right = LANDMARK_IDXS_RIGHT_DOMINANT0
        
    @tf.function
    def call(self, data0):
        data0 = tf.cast(data0, tf.float32)
        left_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0.0, 1.0))
        right_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0.0, 1.0))
        left_dominant = left_hand_sum >= right_hand_sum
        
        if left_dominant:
            frames_hands_non_nan_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0.0, 1.0), axis=[1, 2])
            data = tf.gather(data0, self.landmark_idxs_left, axis=1)
        else:
            frames_hands_non_nan_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0.0, 1.0), axis=[1, 2])
            data = tf.gather(data0, self.landmark_idxs_right, axis=1)
            data = tf.concat([-1.0 * tf.expand_dims(data[:, :, 0], axis=-1), tf.expand_dims(data[:, :, 1], axis=-1), tf.expand_dims(data[:, :, 2], axis=-1)], axis=-1)
            
        non_empty_frames_idxs = tf.where(frames_hands_non_nan_sum > 0)
        non_empty_frames_idxs = tf.squeeze(non_empty_frames_idxs, axis=1)
        data = tf.gather(data, non_empty_frames_idxs, axis=0)

        lips = data[:, :40, :] 
        lips_mean = tf.math.reduce_mean(tf.where(tf.math.is_nan(lips), 0.0, lips), axis=1, keepdims=True)
        lips_std = tf.math.reduce_std(tf.where(tf.math.is_nan(data), 0.0, data), axis=[1,2], keepdims=True) + 1e-6
        data = (data - lips_mean) / lips_std
        data = tf.where(tf.math.is_nan(data), 0.0, data)

        N_FRAMES = tf.shape(data)[0]
        if N_FRAMES < INPUT_SIZE:
            non_empty_frames_idxs = tf.pad(tf.cast(non_empty_frames_idxs, tf.float32), [[0, INPUT_SIZE - N_FRAMES]], constant_values=-1)
            data = tf.pad(data, [[0, INPUT_SIZE - N_FRAMES], [0,0], [0,0]], constant_values=0)
        else:
            data_flat = tf.reshape(data, [1, N_FRAMES, -1, 1])
            data_resized = tf.image.resize(data_flat, [INPUT_SIZE, tf.shape(data_flat)[2]], method=tf.image.ResizeMethod.BILINEAR)
            data = tf.reshape(data_resized, [INPUT_SIZE, -1, 3])
            non_empty_frames_idxs = tf.linspace(0.0, tf.cast(N_FRAMES, tf.float32), INPUT_SIZE)

        dx = data[1:, :, :] - data[:-1, :, :]
        dx = tf.concat([tf.zeros_like(data[:1, :, :]), dx], axis=0)
        ddx = dx[1:, :, :] - dx[:-1, :, :]
        ddx = tf.concat([tf.zeros_like(dx[:1, :, :]), ddx], axis=0)
        data = tf.concat([data, dx, ddx], axis=-1)
        data = tf.reshape(data, (INPUT_SIZE, -1))
        
        return data, non_empty_frames_idxs

# MODEL BUILDER
def get_model():
    embed_dim = 192
    num_heads = 4
    ff_dim = embed_dim * 2
    
    frames = tf.keras.layers.Input([INPUT_SIZE, N_COLS_FINAL], dtype=tf.float16, name='frames')
    non_empty_frame_idxs = tf.keras.layers.Input([INPUT_SIZE], dtype=tf.float16, name='non_empty_frame_idxs')
    
    x = MaskingLayer(name='input_masking')([frames, non_empty_frame_idxs])
    x = tf.keras.layers.Dense(embed_dim, use_bias=False, name='stem_conv')(x)
    x = tf.keras.layers.BatchNormalization(momentum=0.95, name='stem_bn')(x)
    
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    
    x = LearnablePositionalEmbedding(INPUT_SIZE, embed_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.2)(x)
    
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    
    x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.2)(x)
    
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.8)(x) 
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='classifier')(x)
    
    model = tf.keras.models.Model(inputs=[frames, non_empty_frame_idxs], outputs=outputs)
    return model

In [4]:
# Clear Session
tf.keras.backend.clear_session()

# Build Model
print("Building model...")
model = get_model()

# Load Weights
print(f"Loading weights from {WEIGHTS_PATH}...")
model.load_weights(WEIGHTS_PATH)
print("✅ Weights loaded successfully.")

# Compile (Optional for inference, however it's good for evaluating metrics)
model.compile(
    loss='categorical_crossentropy', 
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='acc'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_acc'),
    ]
)

Building model...


I0000 00:00:1770533980.437619      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Loading weights from /kaggle/input/hybrid-model/tensorflow2/default/1/hybrid_model.weights.h5...
✅ Weights loaded successfully.


In [5]:
# Load Test Metadata
try:
    test_df, sign_map = get_test_metadata(LABELS_PATH, MAP_PATH)
    print(f"Found {len(test_df)} test samples.")
except Exception as e:
    print(f"Skipping evaluation (Metadata not found): {e}")
    test_df = None

if test_df is not None:
    # Initialize Preprocessor
    preprocess_layer = PreprocessLayer()
    
    # Storage for Metrics
    correct_count = 0
    top5_correct_count = 0
    total_count = 0
    
    # Iterate and Predict
    print("Starting evaluation loop...")
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        # Path Handling
        clean_rel_path = row['path'].replace('test_landmark_files/', '')
        file_path = os.path.join(TEST_DATA_DIR, clean_rel_path)
        
        if not os.path.exists(file_path):
            file_path = os.path.join(TEST_DATA_DIR, os.path.basename(row['path']))
            
        if not os.path.exists(file_path):
            continue # Skip missing files

        try:
            # Load Data
            data = load_relevant_data_subset(file_path)
            
            # Preprocess (Convert to Tensor)
            data_tf = tf.convert_to_tensor(data)
            frames, idxs = preprocess_layer(data_tf)
            
            # Batch Dimension (Add batch size of 1)
            frames = tf.expand_dims(frames, axis=0)
            idxs = tf.expand_dims(idxs, axis=0)
            
            # Predict
            # verbose=0 is crucial to avoid log spam
            preds = model.predict({'frames': frames, 'non_empty_frame_idxs': idxs}, verbose=0)
            
            # Check Accuracy
            true_label = row['sign_ord']
            pred_label = np.argmax(preds)
            top5_labels = np.argsort(preds)[0][-5:]
            
            if pred_label == true_label:
                correct_count += 1
            
            if true_label in top5_labels:
                top5_correct_count += 1
                
            total_count += 1
            
        except Exception as e:
            # print(f"Error on {file_path}: {e}")
            pass

    # Final Report
    if total_count > 0:
        print("="*30)
        print(f"Total Samples Evaluated: {total_count}")
        print(f"Top-1 Accuracy: {correct_count / total_count:.4f}")
        print(f"Top-5 Accuracy: {top5_correct_count / total_count:.4f}")
        print("="*30)
    else:
        print("No samples were successfully processed.")

Found 39443 test samples.
Starting evaluation loop...


  0%|          | 0/39443 [00:00<?, ?it/s]

I0000 00:00:1770533989.161048     100 service.cc:152] XLA service 0x7fce9c732ba0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770533989.161086     100 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1770533989.475076     100 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1770533992.103544     100 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Total Samples Evaluated: 39443
Top-1 Accuracy: 0.7747
Top-5 Accuracy: 0.9178
